# 01 — Exploration et nettoyage des prix des céréales au Burkina Faso

**Source :** WFP Food Prices for Burkina Faso (HDX — data.humdata.org)

**Objectif de ce notebook :**
1. Charger les données brutes
2. Comprendre leur structure (colonnes, produits, marchés, unités, période)
3. Nettoyer et filtrer les céréales de base
4. Exporter un fichier propre pour l'analyse et Power BI

## 1. Imports et configuration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Affichage de toutes les colonnes dans les aperçus
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Chemins du projet (relatifs au dossier notebooks/)
CHEMIN_BRUT = "../data/raw/wfp_food_prices_bfa.csv"
CHEMIN_PROPRE = "../data/clean/prix_cereales_clean.csv"

## 2. Chargement des données

La ligne 2 du fichier WFP contient des tags HXL (`#date`, `#adm1+name`…), pas des données : on la saute avec `skiprows=[1]`.

In [ ]:
df_brut = pd.read_csv(CHEMIN_BRUT, skiprows=[1])

print("Dimensions :", df_brut.shape)
df_brut.head()

## 3. Exploration de la structure

In [ ]:
# Types des colonnes et valeurs manquantes
df_brut.info()

In [ ]:
# Liste des colonnes
df_brut.columns.tolist()

In [ ]:
# Période couverte
dates = pd.to_datetime(df_brut["date"])
print("Du", dates.min().date(), "au", dates.max().date())

In [ ]:
# Produits disponibles (combien de relevés par produit)
df_brut["commodity"].value_counts()

In [ ]:
# Catégories de produits
df_brut["category"].value_counts()

In [ ]:
# Types de prix (détail / gros) et unités
print(df_brut["pricetype"].value_counts(), "\n")
print(df_brut["unit"].value_counts())

In [ ]:
# Régions et nombre de marchés
print("Nombre de régions :", df_brut["admin1"].nunique())
print("Nombre de marchés :", df_brut["market"].nunique(), "\n")
df_brut["admin1"].value_counts()

## 4. Nettoyage

⚠️ Avant de lancer cette partie, vérifie les résultats de la section 3 :
- les noms exacts des céréales dans `commodity`
- les unités utilisées (KG, 100 KG…)

Adapte `MOTS_CEREALES` si besoin.

In [ ]:
df = df_brut.copy()

# Conversion de la date
df["date"] = pd.to_datetime(df["date"])

# On ne garde que les prix de détail
df = df[df["pricetype"] == "Retail"]

# On garde les céréales de base (recherche par mot-clé, insensible à la casse)
MOTS_CEREALES = ["millet", "maize", "sorghum", "rice"]
masque = df["commodity"].str.lower().str.contains("|".join(MOTS_CEREALES))
df = df[masque]

print("Lignes restantes :", len(df))
df["commodity"].value_counts()

In [ ]:
# Vérification des unités par produit (elles doivent être homogènes pour comparer)
df.groupby(["commodity", "unit"]).size()

In [ ]:
# Monnaie utilisée (normalement XOF = FCFA)
df["currency"].value_counts()

## 5. Contrôle qualité

In [ ]:
# Valeurs manquantes
df.isna().sum()

In [ ]:
# Statistiques des prix par produit : repérer les valeurs aberrantes
df.groupby("commodity")["price"].describe()

In [ ]:
# Visualisation rapide de la distribution des prix
df.boxplot(column="price", by="commodity", figsize=(10, 5), rot=30)
plt.suptitle("")
plt.title("Distribution des prix par produit")
plt.ylabel("Prix (FCFA)")
plt.tight_layout()
plt.show()

## 6. Colonnes utiles pour l'analyse

In [ ]:
df["annee"] = df["date"].dt.year
df["mois"] = df["date"].dt.month

# Période de soudure (juin à septembre) : prix habituellement plus élevés
df["periode"] = df["mois"].apply(lambda m: "Soudure" if 6 <= m <= 9 else "Hors soudure")

df[["date", "annee", "mois", "periode", "admin1", "market", "commodity", "unit", "price"]].head()

## 7. Export du fichier propre

In [ ]:
colonnes = ["date", "annee", "mois", "periode", "admin1", "admin2", "market",
            "latitude", "longitude", "commodity", "unit", "currency", "price", "usdprice"]
colonnes = [c for c in colonnes if c in df.columns]

df[colonnes].to_csv(CHEMIN_PROPRE, index=False)
print(f"Fichier exporté : {CHEMIN_PROPRE} ({len(df)} lignes)")

## 8. Notes et observations

*À compléter au fil de l'exploration :*
- Période couverte :
- Nombre de marchés :
- Problèmes rencontrés (unités, valeurs manquantes…) :
- Premières observations :